# Análise Completa de Dados: Tickets de Atendimento
Este notebook foi desenvolvido para realizar uma análise descritiva e diagnóstica sobre uma base de tickets de atendimento ao cliente.

O objetivo é extrair *insights* valiosos sobre o volume de atendimento, o tempo de resolução, a eficiência dos canais de comunicação, a satisfação do cliente (CSAT) e os custos operacionais.

**Estrutura de Colunas Utilizadas:**
- **Identificadores:** `ticket_id`, `customer_id`, `order_id`
- **Temporalidade:** `data_abertura`, `data_fechamento`
- **Classificação:** `canal_entrada`, `categoria_problema`, `status_atendimento`
- **Qualidade & Custo:** `texto_cliente`, `nota_csat`, `tempo_primeira_resposta_minutos`, `custo_operacional_ticket`


### 1. Importação de Bibliotecas e Carregamento Robusto dos Dados
Nesta etapa, importamos as bibliotecas necessárias (Pandas e Numpy).
Utilizamos um método de leitura robusta para evitar que o DataFrame "quebre" caso algum cliente tenha digitado vírgulas `,` no meio da coluna de texto (`texto_cliente`), o que costuma desalinhar arquivos CSV padrão.


In [4]:
import pandas as pd
import requests
import warnings
warnings.filterwarnings('ignore')

# 1. Pegamos apenas o ID do arquivo que estava no seu link
file_id = '1trJHdY5EQ2hMo9dDgNXXZM00BFw5rIPn'
# 2. Criamos o link oficial de download direto do Google Drive
url_direta = f'https://drive.google.com/uc?export=download&id={file_id}'

# Definimos as 12 colunas esperadas na base
colunas = [
    'ticket_id', 'customer_id', 'order_id', 'data_abertura',
    'data_fechamento', 'canal_entrada', 'categoria_problema',
    'status_atendimento', 'texto_cliente', 'nota_csat',
    'tempo_primeira_resposta_minutos', 'custo_operacional_ticket'
]

linhas_corrigidas = []
linhas_ignoradas = 0

try:
    # 3. Fazemos o download do texto cru do CSV direto do link
    resposta = requests.get(url_direta)
    resposta.raise_for_status() # Verifica se o download funcionou

    # 4. Quebramos o texto gigante baixado em uma lista de linhas
    linhas_arquivo = resposta.text.splitlines()

    if len(linhas_arquivo) > 0:
        header = linhas_arquivo[0] # Ignora a primeira linha (cabeçalho)

        for linha in linhas_arquivo[1:]:
            linha = linha.strip()
            if not linha:
                continue

            partes = linha.split(',')

            # Se a linha dividiu perfeitamente nas 12 colunas:
            if len(partes) == 12:
                linhas_corrigidas.append(partes)

            # Se dividiu em mais de 12 pedaços, significa que há vírgulas sobrando no texto do cliente
            elif len(partes) > 12:
                inicio = partes[:8] # Pega as 8 primeiras colunas corretas
                fim = partes[-3:]   # Pega as 3 últimas colunas corretas
                texto_junto = [",".join(partes[8:-3])] # Junta todo o 'lixo' do meio de volta como um único texto

                linha_refeita = inicio + texto_junto + fim # Reconstrói a linha com 12 posições
                linhas_corrigidas.append(linha_refeita)
            else:
                linhas_ignoradas += 1

    # 5. Criamos o DataFrame
    df = pd.DataFrame(linhas_corrigidas, columns=colunas)
    print(f"✅ Dataset carregado com sucesso! {len(df)} linhas lidas.")

    if linhas_ignoradas > 0:
        print(f"⚠️ Atenção: {linhas_ignoradas} linhas corrompidas foram ignoradas.")

except Exception as e:
    print(f"❌ ERRO ao tentar ler o arquivo do Drive: {e}")

✅ Dataset carregado com sucesso! 35841 linhas lidas.


### 2. Limpeza e Tipagem de Dados
Aqui transformamos textos em formatos adequados: Datas passam a ser do tipo `datetime` e números passam a ser do tipo `float/int`. Também preenchemos campos nulos com informações padrão para não quebrar os cálculos estatísticos.


In [5]:
# 1. Tratamento de Datas
df['data_abertura'] = pd.to_datetime(df['data_abertura'], errors='coerce')
df['data_fechamento'] = pd.to_datetime(df['data_fechamento'], errors='coerce')

# Remover tickets onde não há data de abertura ou ID (são dados inúteis para análise)
df = df.dropna(subset=['data_abertura', 'ticket_id'])

# Criar colunas temporais derivadas (mês-ano e dia)
df['mes'] = df['data_abertura'].dt.to_period('M')
df['dia'] = df['data_abertura'].dt.date

# Calcular o tempo total de resolução (da abertura ao fechamento) em horas
df['tempo_resolucao_horas'] = (df['data_fechamento'] - df['data_abertura']).dt.total_seconds() / 3600

# 2. Preenchimento de Nulos/Vazios (Tratamento de Strings)
df['canal_entrada'] = df['canal_entrada'].fillna('Canal Desconhecido')
df['categoria_problema'] = df['categoria_problema'].fillna('Sem Categoria')
df['texto_cliente'] = df['texto_cliente'].fillna('Sem texto')
df['status_atendimento'] = df['status_atendimento'].fillna('Desconhecido')

# 3. Tratamento Numérico
df['nota_csat'] = pd.to_numeric(df['nota_csat'], errors='coerce')
df['tempo_primeira_resposta_minutos'] = pd.to_numeric(df['tempo_primeira_resposta_minutos'], errors='coerce')
df['custo_operacional_ticket'] = pd.to_numeric(df['custo_operacional_ticket'], errors='coerce').fillna(0)

print(f"🧹 Limpeza concluída! Base pronta para análise com dimensão: {df.shape}")


🧹 Limpeza concluída! Base pronta para análise com dimensão: (35840, 15)


### 3. Análise de Volume e Temporalidade
Nesta seção, identificamos o tamanho da nossa base, o período de tempo que ela cobre e como o volume de tickets se comporta ao longo dos dias da semana e dos meses do ano.


In [6]:
print("--- ANÁLISE DE VOLUME E TEMPORALIDADE ---\n")

# 1. Faixa de tempo analisada
inicio = df['data_abertura'].min().strftime('%d/%m/%Y')
fim = df['data_abertura'].max().strftime('%d/%m/%Y')
print(f"1. Faixa de tempo analisada: De {inicio} até {fim}\n")

# 2. Total de tickets
print(f"2. Tickets totais: {len(df)}\n")

# 3. Distribuição por Dia da Semana
dias_pt = {
    'Monday': 'Segunda-feira', 'Tuesday': 'Terça-feira', 'Wednesday': 'Quarta-feira',
    'Thursday': 'Quinta-feira', 'Friday': 'Sexta-feira', 'Saturday': 'Sábado', 'Sunday': 'Domingo'
}
df['dia_semana'] = df['data_abertura'].dt.day_name().map(dias_pt)
tickets_dia_semana = df['dia_semana'].value_counts()
print(f"3. Tickets por dia da semana (Volume absoluto):\n{tickets_dia_semana}\n")

# 4. Distribuição por Mês (Listagem completa)
tickets_por_mes = df.groupby('mes').size()
print(f"4. Evolução Mensal (Tickets gerados por mês):\n{tickets_por_mes}\n")


--- ANÁLISE DE VOLUME E TEMPORALIDADE ---

1. Faixa de tempo analisada: De 01/01/2023 até 31/12/2025

2. Tickets totais: 35840

3. Tickets por dia da semana (Volume absoluto):
dia_semana
Segunda-feira    5389
Sábado           5098
Quarta-feira     5096
Sexta-feira      5088
Domingo          5066
Terça-feira      5065
Quinta-feira     5038
Name: count, dtype: int64

4. Evolução Mensal (Tickets gerados por mês):
mes
2023-01     680
2023-02     661
2023-03    1264
2023-04     798
2023-05    1327
2023-06     729
2023-07     755
2023-08     780
2023-09     863
2023-10     860
2023-11    1840
2023-12    1420
2024-01     712
2024-02     688
2024-03    1266
2024-04     811
2024-05    1287
2024-06     735
2024-07     800
2024-08     773
2024-09     862
2024-10     822
2024-11    1933
2024-12    1427
2025-01     726
2025-02     700
2025-03    1223
2025-04     745
2025-05    1274
2025-06     805
2025-07     739
2025-08     779
2025-09     819
2025-10     862
2025-11    1679
2025-12    1396
Freq: 

### 4. Análise de Clientes (Tickets Únicos vs. Recorrência)
É fundamental entender se o nosso alto volume de tickets é derivado de muitos clientes novos, ou se temos uma base de clientes que entra em contato repetidas vezes (indicativo de problema não resolvido ou de alta dependência do suporte).


In [7]:
print("--- ANÁLISE DE CLIENTES E RECORRÊNCIA ---\n")

# Clientes distintos
print(f"5. Clientes distintos (Total absoluto de CPFs/IDs): {df['customer_id'].nunique()}\n")

# Separando clientes de ticket único vs clientes recorrentes
rep_clientes = df.groupby('customer_id').size()

clientes_unicos = rep_clientes[rep_clientes == 1]
clientes_recorrentes = rep_clientes[rep_clientes > 1]

tickets_unicos_total = clientes_unicos.sum()
tickets_recorrentes_total = clientes_recorrentes.sum()

print("6. Perfil de Abertura de Chamados:")
print(f"   -> Clientes de Ticket Único: {len(clientes_unicos)} pessoas")
print(f"      (Representando {tickets_unicos_total} tickets na base)\n")
print(f"   -> Clientes Recorrentes (>1 ticket): {len(clientes_recorrentes)} pessoas")
print(f"      (Representando um total massivo de {tickets_recorrentes_total} tickets na base)\n")

# Média de tickets entre os clientes recorrentes
if len(clientes_recorrentes) > 0:
    print(f"   -> Média de tickets gerados por cada cliente recorrente: {(tickets_recorrentes_total / len(clientes_recorrentes)):.2f} tickets/cliente")

# Ranking dos clientes que mais abrem tickets
top_clientes_recorrentes = clientes_recorrentes.sort_values(ascending=False)
print(f"\n7. Top 10 Clientes que mais geraram tickets (Gargalos de Suporte):\n{top_clientes_recorrentes.head(10)}")


--- ANÁLISE DE CLIENTES E RECORRÊNCIA ---

5. Clientes distintos (Total absoluto de CPFs/IDs): 445

6. Perfil de Abertura de Chamados:
   -> Clientes de Ticket Único: 153 pessoas
      (Representando 153 tickets na base)

   -> Clientes Recorrentes (>1 ticket): 292 pessoas
      (Representando um total massivo de 35687 tickets na base)

   -> Média de tickets gerados por cada cliente recorrente: 122.22 tickets/cliente

7. Top 10 Clientes que mais geraram tickets (Gargalos de Suporte):
customer_id
CLI-11830    14355
CLI-01729     7457
CLI-14211     2101
CLI-12360     1302
CLI-03910     1190
CLI-02474      804
CLI-06160      732
CLI-07163      619
CLI-10455      586
CLI-13668      435
dtype: int64


### 5. Análise de Tempo de Resolução (Abertura até Fechamento)
Vamos analisar quanto tempo a operação leva para resolver um chamado.
**Regra de Ouro:** Filtramos *apenas* os tickets com status de 'Resolvido' e ignoramos datas de fechamento fictícias (como datas empurradas para o final de 2025 pelo sistema) para que a nossa média reflita a pura realidade operacional.


In [8]:
print("--- ANÁLISE DE TEMPO DE RESOLUÇÃO ---\n")

# Filtrando dados reais (Status Resolvido e sem a data fictícia de fim de 2025)
data_ficticia = pd.to_datetime('2025-12-31 23:59:00')
df_fechados = df[
    (df['status_atendimento'] == 'Resolvido') &
    (df['data_fechamento'] != data_ficticia)
]

# Tempo médio geral
tempo_medio_fechamento = df_fechados['tempo_resolucao_horas'].mean()
print(f"8. Tempo médio GERAL de resolução: {tempo_medio_fechamento:.2f} horas (aproximadamente {tempo_medio_fechamento/24:.2f} dias)\n")

# Top Meses mais lentos
tempo_por_mes = df_fechados.groupby('mes')['tempo_resolucao_horas'].mean().sort_values(ascending=False)
print(f"   -> Top 5 Meses com MAIOR lentidão no atendimento (em horas):\n{tempo_por_mes.head()}\n")

# Tempo médio por Canal
tempo_por_canal = df_fechados.groupby('canal_entrada')['tempo_resolucao_horas'].mean().sort_values(ascending=False)
print(f"   -> Tempo médio de resolução por Canal de Entrada:\n{tempo_por_canal}\n")

# Tempo médio por Categoria
tempo_por_categoria = df_fechados.groupby('categoria_problema')['tempo_resolucao_horas'].mean().sort_values(ascending=False)
print(f"   -> Tempo médio de resolução por Categoria do Problema:\n{tempo_por_categoria}")


--- ANÁLISE DE TEMPO DE RESOLUÇÃO ---

8. Tempo médio GERAL de resolução: 48.66 horas (aproximadamente 2.03 dias)

   -> Top 5 Meses com MAIOR lentidão no atendimento (em horas):
mes
2023-05    51.060571
2023-02    50.803653
2023-07    50.418511
2025-10    50.377737
2024-09    50.261181
Freq: M, Name: tempo_resolucao_horas, dtype: float64

   -> Tempo médio de resolução por Canal de Entrada:
canal_entrada
Telefone        49.257312
ChatBot         49.204117
Reclame Aqui    48.525026
E-mail          48.453444
WhatsApp        48.318254
Name: tempo_resolucao_horas, dtype: float64

   -> Tempo médio de resolução por Categoria do Problema:
categoria_problema
Defeito                   49.234728
Troca de Tamanho          49.233946
Dúvida Técnica            48.682265
Onde está meu pedido?     48.606363
Elogio                    48.273116
Pagamento não aprovado    47.464799
Name: tempo_resolucao_horas, dtype: float64


### 6. Análises de Canais de Comunicação e Categorias
Vamos cruzar os Canais de Entrada com as Categorias dos problemas e distribuí-los ao longo do tempo (Meses) utilizando Tabelas Pivot (`unstack`). Isso nos permite visualizar facilmente a sazonalidade e a preferência dos clientes.


In [9]:
print("--- ANÁLISE DE CANAIS DE ENTRADA ---\n")

print(f"9. Quantidade de canais: {df['canal_entrada'].nunique()} -> {list(df['canal_entrada'].unique())}")
print(f"\n10. Volume Total Histórico por Canal:\n{df['canal_entrada'].value_counts()}\n")

# Pivot de Canais por Mês
t_canal_mes = df.groupby(['mes', 'canal_entrada']).size().unstack(fill_value=0)

# Média de tickets mensais por canal
media_mensal_canal = t_canal_mes.mean().round(2).sort_values(ascending=False)
print(f"11. Média de volume mensal que cada canal recebe:\n{media_mensal_canal}\n")

print(f"12. Visão Temporal: Tickets por Mês (Linhas) vs Canal (Colunas):\n")
print(t_canal_mes)

print("\n\n--- ANÁLISE DE CATEGORIAS DE PROBLEMA ---\n")
print(f"13. Quantidade de Categorias: {df['categoria_problema'].nunique()} -> {list(df['categoria_problema'].unique())}")
print(f"\n14. Volume Total Histórico por Categoria:\n{df['categoria_problema'].value_counts()}\n")

# Pivot de Categorias por Mês
t_mes_cat = df.groupby(['mes', 'categoria_problema']).size().unstack(fill_value=0)
media_mensal_cat = t_mes_cat.mean().round(2).sort_values(ascending=False)

print(f"15. Média de volume mensal gerado por cada Categoria:\n{media_mensal_cat}\n")
print(f"16. Visão Temporal: Tickets por Mês (Linhas) vs Categoria (Colunas):\n")
print(t_mes_cat)


--- ANÁLISE DE CANAIS DE ENTRADA ---

9. Quantidade de canais: 5 -> ['ChatBot', 'E-mail', 'Telefone', 'WhatsApp', 'Reclame Aqui']

10. Volume Total Histórico por Canal:
canal_entrada
WhatsApp        12651
E-mail           8930
ChatBot          7170
Telefone         4160
Reclame Aqui     2929
Name: count, dtype: int64

11. Média de volume mensal que cada canal recebe:
canal_entrada
WhatsApp        351.42
E-mail          248.06
ChatBot         199.17
Telefone        115.56
Reclame Aqui     81.36
dtype: float64

12. Visão Temporal: Tickets por Mês (Linhas) vs Canal (Colunas):

canal_entrada  ChatBot  E-mail  Reclame Aqui  Telefone  WhatsApp
mes                                                             
2023-01            133     178            66        69       234
2023-02            143     148            41        82       247
2023-03            241     305           126       138       454
2023-04            166     193            62        99       278
2023-05            251     37

### 7. Análise de Status, 1ª Resposta e CSAT (Satisfação)
A CSAT (Customer Satisfaction Score) é a nota dada pelo cliente. Vamos avaliar a média dessa nota cruzada com os Canais e as Categorias, além de avaliar a agilidade da equipe na 1ª Resposta.


In [10]:
print("--- STATUS DE ATENDIMENTO E PRIMEIRA RESPOSTA ---\n")

print(f"17. Distribuição do Status Atual dos Tickets:\n{df['status_atendimento'].value_counts()}\n")

print(f"18. Média Geral de Tempo para a 1ª Resposta: {df['tempo_primeira_resposta_minutos'].mean():.2f} minutos\n")

t_primeira_resp = df.groupby(['canal_entrada', 'categoria_problema'])['tempo_primeira_resposta_minutos'].mean().reset_index()
print(f"19. Média da 1ª Resposta por Canal e Categoria (Amostra):\n{t_primeira_resp.head(10)}\n")


print("\n--- ANÁLISE DE SATISFAÇÃO DO CLIENTE (CSAT) ---\n")

print(f"20. Média de CSAT Geral da Operação: {df['nota_csat'].mean():.2f}\n")

csat_canal = df.groupby('canal_entrada')['nota_csat'].mean().round(2).sort_values(ascending=False)
print(f"-> CSAT por Canal de Entrada (Onde somos mais bem avaliados?):\n{csat_canal}\n")

csat_categoria = df.groupby('categoria_problema')['nota_csat'].mean().round(2).sort_values(ascending=False)
print(f"-> CSAT por Categoria de Problema (Que problema deixa o cliente mais irritado?):\n{csat_categoria}\n")

csat_status = df.groupby('status_atendimento')['nota_csat'].mean().round(2).sort_values(ascending=False)
print(f"-> CSAT por Status de Atendimento:\n{csat_status}")


--- STATUS DE ATENDIMENTO E PRIMEIRA RESPOSTA ---

17. Distribuição do Status Atual dos Tickets:
status_atendimento
Resolvido           23352
Em Análise           5304
Aberto               3594
Escalado para N2     3590
Name: count, dtype: int64

18. Média Geral de Tempo para a 1ª Resposta: 135.41 minutos

19. Média da 1ª Resposta por Canal e Categoria (Amostra):
  canal_entrada      categoria_problema  tempo_primeira_resposta_minutos
0       ChatBot                 Defeito                         1.503459
1       ChatBot          Dúvida Técnica                         1.522031
2       ChatBot                  Elogio                         1.419944
3       ChatBot   Onde está meu pedido?                         1.482086
4       ChatBot  Pagamento não aprovado                         1.587035
5       ChatBot        Troca de Tamanho                         1.506047
6        E-mail                 Defeito                       271.718978
7        E-mail          Dúvida Técnica           

### 8. Análise de Custo Operacional e Insights Finais
Uma operação de atendimento tem custos. Onde estamos a gastar mais dinheiro? Analisamos o custo segregado por canal, categoria e status.


In [11]:
print("--- ANÁLISE FINANCEIRA (CUSTO OPERACIONAL) ---\n")

custo_geral = df['custo_operacional_ticket'].sum()
print(f"21. Custo Operacional Total: R$ {custo_geral:,.2f}\n")

custo_mes = df.groupby('mes')['custo_operacional_ticket'].sum()
print(f"22. Evolução do Custo Operacional por Mês:\n{custo_mes.head()}\n")

print("\n--- DETALHAMENTO DE CUSTOS (RANKING) ---\n")

custo_canal = df.groupby('canal_entrada')['custo_operacional_ticket'].sum().sort_values(ascending=False)
print(f"23a. Custo Total por Canal (Onde gastamos mais):\n{custo_canal}\n")

custo_categoria = df.groupby('categoria_problema')['custo_operacional_ticket'].sum().sort_values(ascending=False)
print(f"23b. Custo Total por Categoria (Qual problema é mais caro de resolver):\n{custo_categoria}\n")

custo_status = df.groupby('status_atendimento')['custo_operacional_ticket'].sum().sort_values(ascending=False)
print(f"23c. Custo Total alocado por Status Atual:\n{custo_status}\n")


print("\n\n--- 💡 INSIGHTS ESTRATÉGICOS (EFICIÊNCIA) ---\n")
# Insight: Relação entre o Custo que um problema gera vs a Nota (CSAT) que o cliente nos dá
eficiencia = df.groupby('categoria_problema').agg({
    'nota_csat': 'mean',
    'custo_operacional_ticket': 'mean'
}).rename(columns={'custo_operacional_ticket': 'custo_medio_por_ticket'}).sort_values(by='nota_csat')

print("Análise de Eficiência Financeira:")
print("-> Ajuda a identificar se estamos a gastar o orçamento justamente nas categorias que o cliente nos avalia mal (Baixo Retorno).")
print(f"\n{eficiencia}")


--- ANÁLISE FINANCEIRA (CUSTO OPERACIONAL) ---

21. Custo Operacional Total: R$ 532,260.00

22. Evolução do Custo Operacional por Mês:
mes
2023-01    10451.0
2023-02     9286.0
2023-03    19607.0
2023-04    11672.0
2023-05    19462.0
Freq: M, Name: custo_operacional_ticket, dtype: float64


--- DETALHAMENTO DE CUSTOS (RANKING) ---

23a. Custo Total por Canal (Onde gastamos mais):
canal_entrada
WhatsApp        189765.0
E-mail          133950.0
Reclame Aqui    131805.0
Telefone         62400.0
ChatBot          14340.0
Name: custo_operacional_ticket, dtype: float64

23b. Custo Total por Categoria (Qual problema é mais caro de resolver):
categoria_problema
Onde está meu pedido?     159660.0
Defeito                    96592.0
Troca de Tamanho           79325.0
Dúvida Técnica             78888.0
Pagamento não aprovado     63856.0
Elogio                     53939.0
Name: custo_operacional_ticket, dtype: float64

23c. Custo Total alocado por Status Atual:
status_atendimento
Resolvido          

---

Com base nos dados fornecidos para a Vértice Retail, a operação de atendimento está a drenar a rentabilidade da empresa devido a falhas sistémicas, falta de automação real e ineficiência operacional.

Abaixo está o diagnóstico consultivo dividido por áreas críticas de impacto e as recomendações de intervenção.

### 1. A Anomalia Crítica: O "Elefante na Sala" (Retrabalho e Bugs)

A descoberta mais alarmante desta base de dados é a taxa de recorrência irreal.

* **Concentração Absurda:** Apenas 445 clientes geraram 35.840 tickets. Pior ainda, um único cliente (`CLI-11830`) gerou **14.355 tickets** (40% de todo o volume da empresa). O segundo maior gerou 7.457.
* **Diagnóstico:** Isto não é comportamento humano. Trata-se claramente de um erro de sistema (um *loop* de API de algum marketplace abrindo chamados automáticos), um bug no formulário do site, ou um ataque de *spam*.
* **Impacto financeiro:** Assumindo o custo médio de R$ 14,80 por ticket, este erro sistémico de meia dúzia de IDs está a custar centenas de milhares de reais à Vértice Retail.

### 2. O Ralo de Produtividade: "Onde está meu pedido?" (WISMO)

A categoria "Onde está meu pedido?" é o maior ofensor de volume (10.765 tickets) e de custo (R$ 159.660).

* **Diagnóstico:** Esta é uma dúvida puramente transacional (dado logístico de rastreio). O fato de o tempo médio de resolução ser de 48,6 horas indica que a equipa humana está a procurar códigos de rastreio manualmente em sistemas fragmentados (transportadoras, ERP, plataforma de e-commerce) e a responder manualmente aos clientes.
* **CSAT:** A nota de 2.99 reflete a frustração do cliente em ter de esperar dois dias para saber onde está a sua encomenda.

### 3. A Ilusão da Automação (Análise do ChatBot)

Os dados mostram que o ChatBot é uma falsa automação.

* **Primeira Resposta vs. Resolução:** O ChatBot responde em 1,5 minutos (o que é esperado), mas o ticket demora **49,2 horas** para ser resolvido (o mesmo tempo do Telefone).
* **Diagnóstico:** O ChatBot não tem integração transacional. Ele atua apenas como um formulário glorificado: recolhe a dúvida do cliente de forma rápida e atira o ticket para a fila da equipa humana resolver. Ele não tem autonomia para consultar o estado do pedido ou processar uma troca.

### 4. Ineficiência de Canais: WhatsApp e Reclame Aqui

* **WhatsApp (O pior cenário):** É o canal preferido dos clientes (maior volume com 12.651 tickets) e onde se gasta mais dinheiro (R$ 189.765), mas entrega a **pior satisfação da empresa** (CSAT de 3.18). A expectativa de um cliente no WhatsApp é de resposta imediata, mas a primeira resposta demora 8,5 minutos e a resolução leva 48 horas.
* **Reclame Aqui (O paradoxo):** Tem a primeira resposta mais lenta de todas (absurdos 770 minutos / ~12 horas), mas apresenta o **maior CSAT (3.30)**.
* **Diagnóstico:** O alto CSAT no Reclame Aqui indica que, quando o caso chega a este nível (público), a equipa de N2/Escalation resolve o problema "dando tudo" ao cliente (cupons, reembolsos integrais, reenvios sem fricção). Isto salva a nota, mas esmaga as margens de lucro da Vértice Retail.

### 5. Problemas de Produto e Logística

* **Defeito e Troca de Tamanho:** Somadas, estas categorias geram quase 12.000 tickets. "Defeito" tem o pior CSAT da empresa (2.82). Numa marca focada em jovens adultos (que exigem agilidade e qualidade), isto aponta para falhas graves de controlo de qualidade na fábrica/fornecedor e uma tabela de medidas mal calibrada no site.

---

### Recomendações de Ação (Plano de Turnaround)

**Ação Imediata (Estancar o Sangramento)**

1. **Auditoria de TI Urgente:** Bloquear imediatamente o ID `CLI-11830` e os outros top 10 IDs. Investigar o *loop* sistémico que está a inundar a base de tickets. Isto cortará instantaneamente mais de 50% do volume e custo operacional.
2. **Revisão da Tabela de Medidas:** Atualizar urgentemente a interface do site/app com um provador virtual inteligente ou tabela de medidas mais clara para estancar as queixas de "Troca de Tamanho".

**Projetos de IA e Automação (Curto/Médio Prazo)**

1. **Integração Real do ChatBot (Agente de IA):** Conectar o bot via API diretamente à base de dados logística e ao ERP. Quando o cliente perguntar "Onde está o meu pedido?", a IA deve consultar o status e responder em segundos, fechando o ticket na hora (Zero-Touch Resolution). Isto atacará o ralo de R$ 159 mil.
2. **Triagem Inteligente no WhatsApp:** Implementar IA no WhatsApp para resolver devoluções e defeitos. O cliente envia a foto do defeito pelo WhatsApp, a IA analisa a imagem, aprova a troca baseada em regras de negócio e gera a etiqueta de devolução dos correios automaticamente, sem intervenção humana.
3. **Dashboard Unificado para N2:** Eliminar a fragmentação de dados criando um painel único (CRM) onde o atendente veja o histórico de compras, status logístico e interação do cliente numa única tela, reduzindo o tempo de resolução de 48h para poucas horas.